In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
import os
import shutil
import transformers
from packaging import version
import pickle

# Configuration
BANGLABERT_NAME = "csebuetnlp/banglabert"
XLM_ROBERTA_NAME = "xlm-roberta-base"
MAX_LEN = 256 # Restored for better accuracy
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5

class MultiModelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer1, tokenizer2, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer1 = tokenizer1
        self.tokenizer2 = tokenizer2
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding1 = self.tokenizer1.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        encoding2 = self.tokenizer2.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids1': encoding1['input_ids'].flatten(),
            'attention_mask1': encoding1['attention_mask'].flatten(),
            'input_ids2': encoding2['input_ids'].flatten(),
            'attention_mask2': encoding2['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class FeatureEnsembleModel(nn.Module):
    def __init__(self, model1_name, model2_name, num_labels, class_weights=None):
        super(FeatureEnsembleModel, self).__init__()
        self.model1 = AutoModel.from_pretrained(model1_name)
        self.model2 = AutoModel.from_pretrained(model2_name)
        
        hidden_size1 = self.model1.config.hidden_size
        hidden_size2 = self.model2.config.hidden_size
        
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size1 + hidden_size2, num_labels)
        self.num_labels = num_labels
        self.class_weights = class_weights

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, labels=None):
        outputs1 = self.model1(input_ids=input_ids1, attention_mask=attention_mask1)
        features1 = outputs1.last_hidden_state[:, 0, :] 
        
        outputs2 = self.model2(input_ids=input_ids2, attention_mask=attention_mask2)
        features2 = outputs2.last_hidden_state[:, 0, :]
        
        combined_features = torch.cat((features1, features2), dim=1)
        combined_features = self.dropout(combined_features)
        
        logits = self.classifier(combined_features)
        
        loss = None
        if labels is not None:
            if self.class_weights is not None:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            else:
                loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return (loss, logits) if loss is not None else logits

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted')
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    
    # Per-class F1 scores
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)
    
    metrics = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_weighted': f1_weighted,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted
    }
    
    # Add per-class F1 scores
    for i, f1_val in enumerate(f1_per_class):
        metrics[f'f1_class_{i}'] = f1_val
    
    return metrics

def main():
    # --- DATA PATHS ---
    train_path = '/kaggle/input/violence-dataset/train.csv'
    val_path = '/kaggle/input/violence-dataset/validation.csv' 
    test_path = '/kaggle/input/violence-dataset/test.csv'
    
    # Load data
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    
    print(f"Loaded {len(train_df)} train, {len(val_df)} validation, and {len(test_df)} test samples.")

    num_labels = 3 
    
    # Compute class weights to handle imbalance
    weights = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
    class_weights = torch.tensor(weights, dtype=torch.float)

    tokenizer1 = AutoTokenizer.from_pretrained(BANGLABERT_NAME)
    tokenizer2 = AutoTokenizer.from_pretrained(XLM_ROBERTA_NAME)

    train_dataset = MultiModelDataset(train_df.text.to_numpy(), train_df.label.to_numpy(), tokenizer1, tokenizer2, MAX_LEN)
    val_dataset = MultiModelDataset(val_df.text.to_numpy(), val_df.label.to_numpy(), tokenizer1, tokenizer2, MAX_LEN)
    test_dataset = MultiModelDataset(test_df.text.to_numpy(), test_df.label.to_numpy(), tokenizer1, tokenizer2, MAX_LEN)

    model = FeatureEnsembleModel(BANGLABERT_NAME, XLM_ROBERTA_NAME, num_labels, class_weights=class_weights)

    # Handle version compatibility
    eval_strat_key = "eval_strategy" if version.parse(transformers.__version__) >= version.parse("4.41.0") else "evaluation_strategy"

    training_args = TrainingArguments(
        output_dir='./ensemble_results',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        **{eval_strat_key: "epoch"},
        save_strategy="no",
        report_to="none"
    )

    class EnsembleTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(
                input_ids1=inputs.get("input_ids1"),
                attention_mask1=inputs.get("attention_mask1"),
                input_ids2=inputs.get("input_ids2"),
                attention_mask2=inputs.get("attention_mask2"),
                labels=labels
            )
            loss = outputs[0] if isinstance(outputs, tuple) else outputs
            return (loss, outputs) if return_outputs else loss

    trainer = EnsembleTrainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)

    print("Starting training...")
    trainer.train()

    print("Evaluating on test set...")
    test_results = trainer.evaluate(test_dataset)
    print(f"Test Results: {test_results}")

    # --- DISK CLEANUP BEFORE SAVE ---
    print("Cleaning up temporary files to free disk space...")
    if os.path.exists('./ensemble_results'):
        shutil.rmtree('./ensemble_results')
    if os.path.exists('./logs'):
        shutil.rmtree('./logs')

    # Save ONLY the final model weights
    print("Saving final model weights...")
    model.to('cpu') # Move to CPU to avoid memory spikes
    
    # Save as .pt file
    torch.save(model.state_dict(), "feature_ensemble_model.pt")
    print("Model saved to feature_ensemble_model.pt")
    
    # Save as .pkl file
    with open("feature_ensemble_model.pkl", "wb") as f:
        pickle.dump(model.state_dict(), f)
    print("Model saved to feature_ensemble_model.pkl")
    
    print("Success! Model saved in both formats.")

if __name__ == "__main__":
    main()

2025-12-30 15:23:25.793024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767108205.980965      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767108206.033180      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767108206.481717      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767108206.481765      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767108206.481768      55 computation_placer.cc:177] computation placer alr

Loaded 8353 train, 1790 validation, and 1790 test samples.


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro,F1 Weighted,Precision Weighted,Recall Weighted,F1 Class 0,F1 Class 1,F1 Class 2
1,0.731000,0.626488,0.755866,0.757178,0.755363,0.767179,0.757014,0.767185,0.755866,0.773692,0.718850,0.778993
2,0.417800,0.558155,0.820112,0.815256,0.812098,0.819782,0.820318,0.821620,0.820112,0.856205,0.783362,0.806202
3,0.266700,0.625275,0.833520,0.831765,0.831057,0.832958,0.833971,0.834960,0.833520,0.857708,0.800000,0.837587
4,0.221500,0.806197,0.836313,0.833407,0.836564,0.830658,0.836168,0.836372,0.836313,0.862069,0.802048,0.836105
5,0.066000,0.887160,0.846927,0.844201,0.849848,0.839807,0.846405,0.847173,0.846927,0.868966,0.815009,0.848629


Evaluating on test set...


Test Results: {'eval_loss': 0.958599865436554, 'eval_accuracy': 0.8324022346368715, 'eval_f1_macro': 0.8285875442849285, 'eval_precision_macro': 0.8349311513105931, 'eval_recall_macro': 0.8238682663086571, 'eval_f1_weighted': 0.8317601640956895, 'eval_precision_weighted': 0.8328223926910733, 'eval_recall_weighted': 0.8324022346368715, 'eval_f1_class_0': 0.8546475358702433, 'eval_f1_class_1': 0.8077260755048288, 'eval_f1_class_2': 0.8233890214797136, 'eval_runtime': 27.1624, 'eval_samples_per_second': 65.9, 'eval_steps_per_second': 4.123, 'epoch': 5.0}
Cleaning up temporary files to free disk space...
Saving final model weights...
Model saved to feature_ensemble_model.pt
Model saved to feature_ensemble_model.pkl
Success! Model saved in both formats.
